In [ ]:
# general imports
import pandas as pd
import os
import json

In [ ]:
# Optional environment setup.
# Set these before initializing vLLM, PyTorch, sentence-transformers, or transformers models.

# Restrict this notebook process to specific GPUs.
# Examples: "0" for one GPU, "0,1" for two GPUs.
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Optional: set for higher Hugging Face Hub rate limits and faster first-time downloads.
# os.environ["HF_TOKEN"] = "hf_..."

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [ ]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)

### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

### Remote mode

Run this cell instead of the local-mode cell if you want to send trial summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.

In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.trials import summarize_trials

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# process = start_vllm_server(task="trial")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
# process.terminate()

### View results

In [ ]:
trial_results.head()

Metadata contains a snapshot of the config used and metadata on the model used. 

In [ ]:
metadata

QC report on the trial summarization run

In [ ]:
qc_report

Save checked-in trial example artifacts


In [ ]:
trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in initial notes

In [ ]:
notes_path = "data/note_set_1.csv"
notes = pd.read_csv(notes_path)

### Local mode

`summarize_patients(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

### Remote mode

Run this cell instead of the local-mode cell if you want to send patient summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.


In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.patients import summarize_patients

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
#process = start_vllm_server(task="patient")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

patient_summaries, metadata, qc_report = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
#process.terminate()

In [ ]:
patient_summaries.head()

### Updating existing patient summaries

Run this optional cell after the local- or remote-mode cell above if you want to simulate a longitudinal update. It reads a second set of new notes, uses the existing `patient_summaries` as the starting point, then updates those summaries with the new notes.


In [ ]:
from matchminer_ai.patients import summarize_patients

new_notes_path = "data/note_set_2.csv"
new_notes = pd.read_csv(new_notes_path)

existing_summaries = patient_summaries.assign(
    patient_summary=lambda df: (
        df["cancer_history_summary"].fillna("")
        + "\n\nBoilerplate:\n"
        + df["general_exclusion_criteria_evidence"].fillna("")
    )
)[["patient_id", "patient_summary"]]

patient_summaries, metadata, qc_report = summarize_patients(
    new_notes,
    config=config,
    existing_summaries=existing_summaries,
    return_metadata=True,
    return_qc=True,
)

### View results

In [ ]:
patient_summaries

Metadata contains a snapshot of the config used and metadata on the models used. 

In [ ]:
metadata

QC report on patient summarization

In [ ]:
qc_report

Save checked-in patient example artifacts

In [ ]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [ ]:
from matchminer_ai.embedding import embed_for_matching

In [ ]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [ ]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

In [ ]:
embedded_trials.head()

Metadata contains a snapshot of the config used and metadata on the models used. 

In [ ]:
metadata

### Patients

In [ ]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [ ]:
embedded_patients.head()

Metadata contains a snapshot of the config used and metadata on the models used. 

In [ ]:
metadata

In [ ]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches

Generate ranked trial-space candidates for each patient.


In [ ]:
from matchminer_ai.matching import generate_candidate_matches

In [ ]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Candidate matches


In [ ]:
candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [ ]:
candidate_matches.head()

## 5. Match quality check


In [ ]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the match-quality checker.


In [ ]:
candidate_matches_with_context = (
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
candidate_matches_with_context.head()

Run the match-quality checker.


In [ ]:
match_quality_matches, metadata = score_match_quality(
    candidate_matches_with_context, return_metadata=True
)

In [ ]:
match_quality_matches.head()

In [ ]:
metadata

## 6. Exclusion criteria check


In [ ]:
# limit to unique patient-trial ID pairs
match_quality_matches["trial_id"] = (
    match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

In [ ]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

In [ ]:
exclusion_results["exclusion_criteria_pass"].value_counts()

In [ ]:
metadata

## 7. Save final matches

Create one review table with one row per patient/trial pair. Each pair is represented by its top-scoring clinical space, rows are ranked by that top match-quality score, and exclusion-check results are included as labels without filtering rows.


In [ ]:
patient_context_cols = [
    col
    for col in [
        "patient_id",
        "cancer_history_summary",
        "general_exclusion_criteria_evidence",
    ]
    if col in patient_summaries.columns
]
trial_context_cols = [
    col
    for col in [
        "space_trial_id",
        "clinical_space_number",
        "clinical_space_summary",
        "general_exclusion_criteria",
    ]
    if col in trial_results.columns
]
original_trial_cols = [
    col
    for col in [
        "trial_id",
        "oncore_id",
        "trial_title",
        "brief_summary",
        "detailed_summary",
        "eligibility_criteria",
    ]
    if col in trials.columns
]

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score", "rank"]],
        on=["patient_id", "space_trial_id"],
        how="left",
    )
    .merge(
        exclusion_results,
        on=["patient_id", "trial_id"],
        how="left",
    )
    .merge(
        patient_summaries[patient_context_cols],
        on="patient_id",
        how="left",
    )
    .merge(
        trial_results[trial_context_cols].drop_duplicates("space_trial_id"),
        on="space_trial_id",
        how="left",
    )
    .merge(
        trials[original_trial_cols].drop_duplicates("trial_id"),
        on="trial_id",
        how="left",
    )
)

# Keep one row per patient/trial pair, represented by the top-scoring clinical space.
final_matches = (
    final_matches.sort_values(
        ["patient_id", "trial_id", "match_quality_score"],
        ascending=[True, True, False],
    )
    .drop_duplicates(["patient_id", "trial_id"], keep="first")
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches["match_quality_rank"] = final_matches.groupby("patient_id").cumcount() + 1

final_matches = final_matches.rename(
    columns={
        "match_quality_rank": "match_rank",
        "exclusion_criteria_pass": "passes_exclusion_criteria",
        "cancer_history_summary": "patient_summary",
        "general_exclusion_criteria_evidence": "patient_exclusion_evidence",
        "brief_summary": "trial_brief_summary",
        "detailed_summary": "trial_detailed_summary",
        "eligibility_criteria": "trial_eligibility_criteria",
        "clinical_space_summary": "matched_clinical_space_summary",
        "general_exclusion_criteria": "extracted_trial_exclusion_criteria",
    }
)

ordered_cols = [
    "patient_id",
    "trial_id",
    "match_rank",
    "passes_exclusion_criteria",
    "patient_summary",
    "patient_exclusion_evidence",
    "trial_title",
    "trial_brief_summary",
    "trial_eligibility_criteria",
    "clinical_space_number",
    "matched_clinical_space_summary",
    "extracted_trial_exclusion_criteria",
]
final_matches = final_matches[[col for col in ordered_cols if col in final_matches.columns]]
final_matches.head()


In [ ]:
# Optional local export for review or downstream analysis.
final_matches.to_csv("output/final_matches.csv", index=None)